# 市場情緒監控面板 Market Sentiment Dashboard

Data source: **FRED (Federal Reserve Economic Data)**
Get a free API key at: https://fred.stlouisfed.org/docs/api/api_key.html

Indicators covered:
- **Fear & Greed Composite Index** (6 components)
- **VIX** — CBOE Volatility Index
- **S&P 500 Technical Analysis** — RSI, MACD, Bollinger Bands, MA50/MA200
- **Credit Spreads** — High Yield OAS, Investment Grade OAS
- **Consumer & Labor Sentiment** — UMich Consumer Sentiment, Jobless Claims
- **Yield Curve & Macro** — 10Y-2Y Spread, Breakeven Inflation, Fed Funds Rate

In [ ]:
%pip install --quiet fredapi pandas numpy matplotlib seaborn

In [ ]:
# ============================================================
# SET YOUR FRED API KEY HERE
# Free registration: https://fred.stlouisfed.org/docs/api/api_key.html
# ============================================================
FRED_API_KEY = "YOUR_API_KEY_HERE"

In [ ]:
from fredapi import Fred
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'grid.alpha': 0.8,
    'legend.facecolor': '#21262d',
    'legend.edgecolor': '#30363d',
})

END_DATE   = datetime.today()
START_DATE = END_DATE - timedelta(days=365 * 2)   # 2 years of history
LOOKBACK   = 252                                   # ~1 trading year for percentile scores
print(f"Date range: {START_DATE.date()} → {END_DATE.date()}")

In [ ]:
fred = Fred(api_key=FRED_API_KEY)

SERIES = {
    'SP500':      ('SP500',           'S&P 500 Index'),
    'VIX':        ('VIXCLS',          'CBOE Volatility Index'),
    'HY_SPREAD':  ('BAMLH0A0HYM2',    'US High Yield OAS (bps)'),
    'IG_SPREAD':  ('BAMLC0A0CM',      'US Investment Grade OAS (bps)'),
    'UMCSENT':    ('UMCSENT',         'UMich Consumer Sentiment'),
    'ICSA':       ('ICSA',            'Initial Jobless Claims'),
    'T10Y2Y':     ('T10Y2Y',          '10Y-2Y Treasury Spread (%)'),
    'T10YIE':     ('T10YIE',          '10Y Breakeven Inflation (%)'),
    'DGS10':      ('DGS10',           '10Y Treasury Yield (%)'),
    'DGS2':       ('DGS2',            '2Y Treasury Yield (%)'),
    'FEDFUNDS':   ('FEDFUNDS',        'Federal Funds Rate (%)'),
}

raw = {}
print("Fetching FRED series...")
for key, (sid, label) in SERIES.items():
    try:
        s = fred.get_series(sid, observation_start=START_DATE, observation_end=END_DATE)
        raw[key] = s
        print(f"  ✓ {label:40s} {len(s):4d} obs")
    except Exception as e:
        print(f"  ✗ {label:40s} ERROR: {e}")
        raw[key] = pd.Series(dtype=float)

# Align all series to a business-day index with forward-fill
idx = pd.bdate_range(START_DATE, END_DATE)
df  = pd.DataFrame({k: v.reindex(idx).ffill() for k, v in raw.items()})
df.index.name = 'Date'

print(f"\nAligned DataFrame: {df.shape[0]} rows × {df.shape[1]} cols")
df.tail(3)

In [ ]:
# ── Technical helpers ────────────────────────────────────────────────────────

def calc_rsi(s, window=14):
    delta = s.diff()
    up    = delta.clip(lower=0)
    down  = -delta.clip(upper=0)
    rs    = up.ewm(com=window-1, min_periods=window).mean() / \
            down.ewm(com=window-1, min_periods=window).mean()
    return 100 - (100 / (1 + rs))

def calc_macd(s, fast=12, slow=26, signal=9):
    macd = s.ewm(span=fast, adjust=False).mean() - s.ewm(span=slow, adjust=False).mean()
    sig  = macd.ewm(span=signal, adjust=False).mean()
    return macd, sig, macd - sig

def calc_bbands(s, window=20, k=2):
    m = s.rolling(window).mean()
    d = s.rolling(window).std()
    return m + k*d, m, m - k*d

def pct_rank(s, window=252, invert=False):
    # rolling percentile rank [0,100]; invert=True makes high values score low
    rank = s.rolling(window).apply(
        lambda x: pd.Series(x).rank(pct=True).iloc[-1] * 100, raw=False
    )
    return (100 - rank) if invert else rank

# ── SP500 technicals ─────────────────────────────────────────────────────────
sp       = df['SP500'].dropna()
rsi_sp   = calc_rsi(sp)
macd_l, macd_s, macd_h = calc_macd(sp)
bb_u, bb_m, bb_l = calc_bbands(sp)
ma50     = sp.rolling(50).mean()
ma200    = sp.rolling(200).mean()
hv30     = sp.pct_change().rolling(30).std() * np.sqrt(252) * 100

# ── Fear & Greed composite (6 equally-weighted components, each 0–100) ────────
fg = {
    'VIX':              pct_rank(df['VIX'].dropna(),       LOOKBACK, invert=True),
    'HY Spread':        pct_rank(df['HY_SPREAD'].dropna(), LOOKBACK, invert=True),
    'Consumer Sent.':   pct_rank(df['UMCSENT'].dropna(),   LOOKBACK, invert=False),
    'SP500 RSI':        rsi_sp,
    'Yield Curve':      pct_rank(df['T10Y2Y'].dropna(),    LOOKBACK, invert=False),
    'Price vs MA200':   pct_rank(((sp - ma200) / ma200 * 100).clip(-30, 30).dropna(),
                                 LOOKBACK, invert=False),
}
fg_df        = pd.DataFrame(fg).dropna()
fg_composite = fg_df.mean(axis=1)

latest_fg = float(fg_composite.iloc[-1]) if not fg_composite.empty else 50.0
fg_zones  = [(0,25,'Extreme Fear','#ef5350'), (25,45,'Fear','#ff7043'),
             (45,55,'Neutral','#ffca28'),     (55,75,'Greed','#66bb6a'),
             (75,101,'Extreme Greed','#00e676')]
fg_label, fg_color = next(
    (lbl, clr) for lo, hi, lbl, clr in fg_zones if lo <= latest_fg < hi
)
print(f"Fear & Greed: {latest_fg:.1f}/100  →  {fg_label}  ({fg_color})")
print("\nLatest component scores:")
for col in fg_df.columns:
    print(f"  {col:20s} {fg_df[col].iloc[-1]:.1f}")

In [ ]:
C = dict(
    price='#4fc3f7', ma50='#ffb74d', ma200='#f06292', bb='#607d8b',
    bull='#66bb6a',  bear='#ef5350', neutral='#ffca28',
    vix='#ce93d8',   hy='#ff8a65',  ig='#80cbc4',
    sent='#aed581',  jobless='#ff7043',
    inf='#e91e63',   ff='#03a9f4',
    grid='#21262d',
)

fig = plt.figure(figsize=(22, 32), facecolor='#0d1117')
fig.suptitle(
    f"Market Sentiment Dashboard  ·  {END_DATE.strftime('%Y-%m-%d')}",
    fontsize=19, fontweight='bold', color='white', y=0.99,
)

gs = gridspec.GridSpec(
    7, 2, figure=fig,
    hspace=0.50, wspace=0.32,
    height_ratios=[1.5, 1.1, 1.5, 0.85, 0.85, 1.1, 1.1],
)

LAST252 = slice(-252, None)
LAST180 = slice(-180, None)

# ── 0L: Fear & Greed gauge ───────────────────────────────────────────────────
ax_g = fig.add_subplot(gs[0, 0])
theta = np.linspace(np.pi, 0, 500)
for lo, hi, _, clr in fg_zones:
    mask = (theta >= np.pi * (1 - hi/100)) & (theta <= np.pi * (1 - lo/100))
    ax_g.fill_between(np.cos(theta[mask]), np.zeros(mask.sum()),
                      np.sin(theta[mask]), color=clr, alpha=0.85)
angle = np.pi * (1 - latest_fg / 100)
ax_g.annotate('', xy=(0.58*np.cos(angle), 0.58*np.sin(angle)), xytext=(0,0),
              arrowprops=dict(arrowstyle='->', color='white', lw=3.5))
ax_g.set_xlim(-1.15, 1.15); ax_g.set_ylim(-0.25, 1.1)
ax_g.set_aspect('equal'); ax_g.axis('off')
ax_g.text(0, -0.13, f'{latest_fg:.0f}', ha='center', fontsize=32,
          fontweight='bold', color=fg_color)
ax_g.text(0, -0.24, fg_label, ha='center', fontsize=14, color=fg_color)
for txt, x in zip(['Extreme\nFear','Fear','Neutral','Greed','Extreme\nGreed'],
                  [-0.95, -0.55, 0, 0.55, 0.95]):
    ax_g.text(x, -0.05, txt, ha='center', fontsize=6.5, color='#8b949e')
ax_g.set_title('Fear & Greed Index', color='white', pad=8, fontsize=13)

# ── 0R: Fear & Greed timeline ────────────────────────────────────────────────
ax_fgl = fig.add_subplot(gs[0, 1])
fgp = fg_composite.iloc[LAST180]
ax_fgl.fill_between(fgp.index, fgp.values, 50,
                    where=fgp.values > 50, color=C['bull'], alpha=0.35)
ax_fgl.fill_between(fgp.index, fgp.values, 50,
                    where=fgp.values <= 50, color=C['bear'], alpha=0.35)
ax_fgl.plot(fgp.index, fgp.values, color='white', lw=1.6)
for lvl, clr, ls in [(75, C['bull'], '--'), (25, C['bear'], '--'), (50, C['neutral'], ':')]:
    ax_fgl.axhline(lvl, color=clr, lw=0.9, ls=ls, alpha=0.7)
ax_fgl.set_ylim(0, 100); ax_fgl.grid(True, color=C['grid'])
ax_fgl.set_title('Fear & Greed — Last 6 Months', color='white', pad=8, fontsize=13)

# ── 1L: VIX ─────────────────────────────────────────────────────────────────
ax_v = fig.add_subplot(gs[1, 0])
vp = df['VIX'].dropna().iloc[LAST252]
ax_v.fill_between(vp.index, vp.values, alpha=0.35, color=C['vix'])
ax_v.plot(vp.index, vp.values, color=C['vix'], lw=1.6)
for lvl, lbl, clr in [(20, 'Fear (20)', C['neutral']), (30, 'Extreme Fear (30)', C['bear'])]:
    ax_v.axhline(lvl, color=clr, lw=1, ls='--', alpha=0.8, label=lbl)
ax_v.set_title(f"VIX  (latest: {vp.iloc[-1]:.2f})", color='white', pad=8, fontsize=13)
ax_v.legend(fontsize=8); ax_v.grid(True, color=C['grid'])

# ── 1R: Historical Volatility ────────────────────────────────────────────────
ax_hv = fig.add_subplot(gs[1, 1])
hvp = hv30.dropna().iloc[LAST252]
ax_hv.plot(hvp.index, hvp.values, color='#b39ddb', lw=1.6)
ax_hv.fill_between(hvp.index, hvp.values, alpha=0.25, color='#b39ddb')
ax_hv.set_title(f"S&P 500 HV30  (latest: {hvp.iloc[-1]:.1f}%)",
                color='white', pad=8, fontsize=13)
ax_hv.set_ylabel('%', color='white'); ax_hv.grid(True, color=C['grid'])

# ── 2: SP500 price + MA + Bollinger ─────────────────────────────────────────
ax_p = fig.add_subplot(gs[2, :])
sp_p   = sp.iloc[LAST252]
bu, bm, bl_ = bb_u.iloc[LAST252], bb_m.iloc[LAST252], bb_l.iloc[LAST252]
ax_p.fill_between(sp_p.index, bl_, bu, alpha=0.1, color=C['bb'])
ax_p.plot(sp_p.index, bu,        color=C['bb'],    lw=0.9, ls='--', label='BB Upper/Lower')
ax_p.plot(sp_p.index, bl_,       color=C['bb'],    lw=0.9, ls='--')
ax_p.plot(sp_p.index, sp_p,      color=C['price'], lw=1.9, label='S&P 500')
ax_p.plot(sp_p.index, ma50.iloc[LAST252],  color=C['ma50'],  lw=1.3, label='MA50')
ax_p.plot(sp_p.index, ma200.iloc[LAST252], color=C['ma200'], lw=1.3, label='MA200')
cross = (ma50.iloc[LAST252] - ma200.iloc[LAST252]).diff()
for dt in cross[cross > 0].index[-3:]:
    ax_p.axvline(dt, color=C['bull'], lw=0.9, ls=':', alpha=0.6)
for dt in cross[cross < 0].index[-3:]:
    ax_p.axvline(dt, color=C['bear'], lw=0.9, ls=':', alpha=0.6)
ax_p.set_title(f"S&P 500  (latest: {sp_p.iloc[-1]:,.0f})", color='white', pad=8, fontsize=13)
ax_p.legend(fontsize=9, loc='upper left'); ax_p.grid(True, color=C['grid'])

# ── 3: RSI ───────────────────────────────────────────────────────────────────
ax_r = fig.add_subplot(gs[3, :])
rp = rsi_sp.iloc[LAST252]
ax_r.fill_between(rp.index, rp.values, 50,
                  where=rp.values > 50, color=C['bull'], alpha=0.3)
ax_r.fill_between(rp.index, rp.values, 50,
                  where=rp.values < 50, color=C['bear'], alpha=0.3)
ax_r.plot(rp.index, rp.values, color='white', lw=1.6)
ax_r.axhline(70, color=C['bull'],    lw=1.1, ls='--', label='Overbought (70)')
ax_r.axhline(30, color=C['bear'],    lw=1.1, ls='--', label='Oversold (30)')
ax_r.axhline(50, color=C['neutral'], lw=0.8, ls=':', alpha=0.6)
ax_r.set_ylim(0, 100)
ax_r.set_title(f"RSI (14)  (latest: {rp.iloc[-1]:.1f})", color='white', pad=8, fontsize=13)
ax_r.legend(fontsize=9, loc='upper right'); ax_r.grid(True, color=C['grid'])

# ── 4: MACD ──────────────────────────────────────────────────────────────────
ax_m = fig.add_subplot(gs[4, :])
ml, ms_, mh = macd_l.iloc[LAST252], macd_s.iloc[LAST252], macd_h.iloc[LAST252]
bar_clr = [C['bull'] if v >= 0 else C['bear'] for v in mh]
ax_m.bar(mh.index, mh.values, color=bar_clr, alpha=0.55, width=1)
ax_m.plot(ml.index, ml.values, color=C['price'],  lw=1.6, label='MACD')
ax_m.plot(ms_.index, ms_.values, color=C['ma50'], lw=1.2, ls='--', label='Signal')
ax_m.axhline(0, color='white', lw=0.8, alpha=0.5)
status = 'Bullish Cross ↑' if ml.iloc[-1] > ms_.iloc[-1] else 'Bearish Cross ↓'
ax_m.set_title(f"MACD (12,26,9)  →  {status}", color='white', pad=8, fontsize=13)
ax_m.legend(fontsize=9, loc='upper left'); ax_m.grid(True, color=C['grid'])

# ── 5L: Credit Spreads ───────────────────────────────────────────────────────
ax_cr = fig.add_subplot(gs[5, 0])
hyp = df['HY_SPREAD'].dropna().iloc[LAST252]
igp = df['IG_SPREAD'].dropna().iloc[LAST252]
ax_cr.plot(hyp.index, hyp.values, color=C['hy'], lw=1.6,
           label=f"HY OAS ({hyp.iloc[-1]:.0f} bps)")
ax_cr2 = ax_cr.twinx()
ax_cr2.plot(igp.index, igp.values, color=C['ig'], lw=1.5, ls='--',
            label=f"IG OAS ({igp.iloc[-1]:.0f} bps)")
ax_cr2.set_ylabel('IG OAS (bps)', color=C['ig'], fontsize=9)
ax_cr2.tick_params(colors=C['ig'])
ax_cr.set_ylabel('HY OAS (bps)', color=C['hy'], fontsize=9)
ax_cr.tick_params(colors=C['hy'])
lns = ax_cr.get_legend_handles_labels()
lns2 = ax_cr2.get_legend_handles_labels()
ax_cr.legend(lns[0]+lns2[0], lns[1]+lns2[1], fontsize=8, loc='upper left')
ax_cr.set_title('Credit Spreads (Fear Indicator)', color='white', pad=8, fontsize=13)
ax_cr.grid(True, color=C['grid'])

# ── 5R: Consumer Sentiment + Jobless Claims ──────────────────────────────────
ax_s = fig.add_subplot(gs[5, 1])
setp = df['UMCSENT'].dropna().iloc[-60:]
jobp = df['ICSA'].dropna().iloc[-104:]
ax_s.plot(setp.index, setp.values, color=C['sent'], lw=1.5, marker='o', ms=3,
          label=f"UMich Sentiment ({setp.iloc[-1]:.1f})")
ax_s2 = ax_s.twinx()
ax_s2.plot(jobp.index, jobp.values/1000, color=C['jobless'], lw=1.3, alpha=0.75,
           label=f"Jobless Claims ({jobp.iloc[-1]/1000:.0f}K)")
ax_s2.set_ylabel('Jobless Claims (K)', color=C['jobless'], fontsize=9)
ax_s2.tick_params(colors=C['jobless'])
ax_s.set_ylabel('Sentiment Index', color=C['sent'], fontsize=9)
ax_s.tick_params(colors=C['sent'])
lns = ax_s.get_legend_handles_labels()
lns2 = ax_s2.get_legend_handles_labels()
ax_s.legend(lns[0]+lns2[0], lns[1]+lns2[1], fontsize=8, loc='lower right')
ax_s.set_title('Consumer Sentiment & Labor Market', color='white', pad=8, fontsize=13)
ax_s.grid(True, color=C['grid'])

# ── 6L: Yield Curve ──────────────────────────────────────────────────────────
ax_yc = fig.add_subplot(gs[6, 0])
ycp = df['T10Y2Y'].dropna().iloc[LAST252]
ax_yc.fill_between(ycp.index, ycp.values, 0,
                   where=ycp.values >= 0, color=C['bull'], alpha=0.35)
ax_yc.fill_between(ycp.index, ycp.values, 0,
                   where=ycp.values < 0,  color=C['bear'], alpha=0.45,
                   label='Inverted (Recession Risk)')
ax_yc.plot(ycp.index, ycp.values, color='white', lw=1.6)
ax_yc.axhline(0, color='white', lw=1.2, ls='--', alpha=0.7)
ax_yc.set_title(f"10Y-2Y Yield Curve  ({ycp.iloc[-1]:+.2f}%)",
                color='white', pad=8, fontsize=13)
ax_yc.legend(fontsize=8); ax_yc.grid(True, color=C['grid'])

# ── 6R: Breakeven Inflation + Fed Funds ─────────────────────────────────────
ax_bk = fig.add_subplot(gs[6, 1])
bkp = df['T10YIE'].dropna().iloc[LAST252]
ffp = df['FEDFUNDS'].dropna().iloc[LAST252]
ax_bk.plot(bkp.index, bkp.values, color=C['inf'], lw=1.6,
           label=f"10Y Breakeven ({bkp.iloc[-1]:.2f}%)")
ax_bk2 = ax_bk.twinx()
ax_bk2.plot(ffp.index, ffp.values, color=C['ff'], lw=1.5, ls='--',
            label=f"Fed Funds Rate ({ffp.iloc[-1]:.2f}%)")
ax_bk2.set_ylabel('Fed Funds Rate (%)', color=C['ff'], fontsize=9)
ax_bk2.tick_params(colors=C['ff'])
ax_bk.set_ylabel('Breakeven Inflation (%)', color=C['inf'], fontsize=9)
ax_bk.tick_params(colors=C['inf'])
lns = ax_bk.get_legend_handles_labels()
lns2 = ax_bk2.get_legend_handles_labels()
ax_bk.legend(lns[0]+lns2[0], lns[1]+lns2[1], fontsize=8, loc='upper left')
ax_bk.set_title('Inflation Expectations & Fed Policy', color='white', pad=8, fontsize=13)
ax_bk.grid(True, color=C['grid'])

plt.savefig('market_sentiment_dashboard.png', dpi=120,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("Saved: market_sentiment_dashboard.png")

In [ ]:
def zone_label(val, thresholds):
    for t, lbl in thresholds:
        if val < t:
            return lbl
    return thresholds[-1][1]

latest = {
    'fg':    latest_fg,
    'vix':   float(df['VIX'].dropna().iloc[-1]),
    'rsi':   float(rsi_sp.dropna().iloc[-1]),
    'sp':    float(sp.iloc[-1]),
    'ma200': float(ma200.dropna().iloc[-1]),
    'hy':    float(df['HY_SPREAD'].dropna().iloc[-1]),
    'yc':    float(df['T10Y2Y'].dropna().iloc[-1]),
    'sent':  float(df['UMCSENT'].dropna().iloc[-1]),
    'bk':    float(df['T10YIE'].dropna().iloc[-1]),
    'ff':    float(df['FEDFUNDS'].dropna().iloc[-1]),
}
pct_ma200 = (latest['sp'] / latest['ma200'] - 1) * 100
macd_status = 'Bullish Cross ↑' if float(macd_l.iloc[-1]) > float(macd_s.iloc[-1]) else 'Bearish Cross ↓'
hy_status   = 'Elevated Risk-Off' if latest['hy'] > 500 else 'Tightening Risk-On'
yc_status   = ('Deeply Inverted ⚠' if latest['yc'] < -0.5 else
               'Inverted'            if latest['yc'] < 0    else 'Normal / Positive')
vix_status  = ('Extreme Fear' if latest['vix'] > 30 else
               'Elevated'     if latest['vix'] > 20 else 'Calm / Low Vol')
rsi_status  = ('Overbought'   if latest['rsi'] > 70 else
               'Oversold'     if latest['rsi'] < 30 else 'Neutral')

sep = '=' * 60
print(sep)
print(f"  MARKET SENTIMENT SUMMARY  ·  {END_DATE.strftime('%Y-%m-%d')}")
print(sep)
print(f"  Fear & Greed Index   : {latest['fg']:>6.1f} / 100   →  {fg_label}")
print(f"  VIX                  : {latest['vix']:>7.2f}        →  {vix_status}")
print(f"  S&P 500 RSI (14)     : {latest['rsi']:>7.1f}        →  {rsi_status}")
print(f"  S&P 500 vs MA200     : {pct_ma200:>+7.2f}%       →  {'Above (Bullish)' if pct_ma200 > 0 else 'Below (Bearish)'}")
print(f"  MACD Signal          :                  →  {macd_status}")
print(f"  HY Credit Spread     : {latest['hy']:>6.0f} bps    →  {hy_status}")
print(f"  Yield Curve 10Y-2Y   : {latest['yc']:>+7.2f}%       →  {yc_status}")
print(f"  Consumer Sentiment   : {latest['sent']:>7.1f}        →  {'High Confidence' if latest['sent'] > 80 else 'Low Confidence'}")
print(f"  10Y Breakeven Infl.  : {latest['bk']:>7.2f}%       →  {'Elevated' if latest['bk'] > 2.5 else 'Anchored'}")
print(f"  Fed Funds Rate       : {latest['ff']:>7.2f}%")
print(sep)

In [ ]:
import subprocess, os

nb_path = os.path.abspath('market_sentiment_dashboard.ipynb')
result  = subprocess.run(
    ['jupyter', 'nbconvert', '--to', 'html', nb_path],
    capture_output=True, text=True,
)
if result.returncode == 0:
    html = nb_path.replace('.ipynb', '.html')
    print(f"HTML export successful:\n  {html}")
else:
    print("Export error:", result.stderr[:500])